# Lab 07: Deployment

## Business Context

Evaluation looks solid. Deploy the agent as a REST API. Run the signature query: *"Top 10 performing stocks with their clarity scores."*

**After this lab:** The agent is live as a Model Serving endpoint. Analysts query it over HTTP. The signature query runs through the deployed endpoint — the answer the whole lab series was building toward.

| Attribute | Detail |
|---|---|
| **Key Skills** | `ServedEntityInput`, `scale_to_zero`, `serving_endpoints`, `TrafficConfig`, `ai_query()`, provisioned vs pay-per-token |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$2-4 |

In [ ]:
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
import json
from databricks.sdk import WorkspaceClient

CATALOG       = "ipo_analyzer"
SCHEMA        = "default"
LLM_ENDPOINT  = "databricks-meta-llama-3.1-405b-instruct"
ENDPOINT_NAME = "ipo-analyzer-endpoint"
MODEL_NAME    = f"{CATALOG}.{SCHEMA}.ipo_filing_agent"

print(f"Catalog        : {CATALOG}")
print(f"Schema         : {SCHEMA}")
print(f"LLM            : {LLM_ENDPOINT}")
print(f"Endpoint name  : {ENDPOINT_NAME}")
print(f"Model name     : {MODEL_NAME}")

w = WorkspaceClient()
print(f"WorkspaceClient: ready (host={w.config.host})")

## A. Deploy to Model Serving

**Model Serving** exposes a Unity Catalog model as a REST endpoint. Clients send HTTP POST requests and receive JSON responses — no Databricks SDK or cluster required on the client side.

Key parameters of `ServedEntityInput`:

| Parameter | Value | Meaning |
|---|---|---|
| `entity_name` | `ipo_analyzer.default.ipo_filing_agent` | Fully-qualified Unity Catalog model name |
| `entity_version` | `"1"` | Model version registered in Lab 03 |
| `workload_size` | `"Small"` | Concurrency tier: Small / Medium / Large |
| `scale_to_zero_enabled` | `True` | Endpoint shuts down after inactivity; no idle cost |

**`scale_to_zero_enabled=True`** is the pay-per-token (serverless) pattern — the endpoint scales down to zero replicas when idle, eliminating cost between requests. The trade-off is a cold-start delay (typically 30–90 seconds) on the first request after idle. For production workloads with consistent traffic, **provisioned throughput** (a dedicated replica count, `scale_to_zero_enabled=False`) eliminates cold starts but incurs a continuous cost.

`create_and_wait()` blocks until the endpoint reaches `READY` state — this can take 5–15 minutes for the initial deploy.

In [ ]:
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
import urllib.request, json, time

# Get latest model version
versions = list(w.model_versions.list(full_name=MODEL_NAME))
latest_version = str(max(int(v.version) for v in versions))
print(f"Latest model version: {latest_version}")

# Get auth from notebook context
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
api_host = ctx.apiUrl().get()
api_token = ctx.apiToken().get()

def api_call(method, path, data=None):
    payload = json.dumps(data).encode('utf-8') if data else None
    req = urllib.request.Request(f'{api_host}{path}', data=payload,
        headers={'Authorization': f'Bearer {api_token}', 'Content-Type': 'application/json'}, method=method)
    try:
        with urllib.request.urlopen(req) as resp: return json.loads(resp.read())
    except urllib.error.HTTPError as e: return {'error': e.read().decode()[:500]}

# Create or update endpoint
try:
    existing = w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Endpoint exists (v{existing.config.served_entities[0].entity_version}). Updating to v{latest_version}...")
    r = api_call('PUT', f'/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config', {
        'served_entities': [{'entity_name': MODEL_NAME, 'entity_version': latest_version,
            'workload_size': 'Small', 'scale_to_zero_enabled': True}]
    })
    if 'error' in r:
        print(f"Update note: {r['error'][:200]}")
    else:
        print("Config update started.")
except Exception:
    print(f"Creating new endpoint with model v{latest_version}...")
    r = api_call('POST', '/api/2.0/serving-endpoints', {
        'name': ENDPOINT_NAME,
        'config': {'served_entities': [{'entity_name': MODEL_NAME, 'entity_version': latest_version,
            'workload_size': 'Small', 'scale_to_zero_enabled': True}]}
    })
    if 'error' in r:
        print(f"Create note: {r['error'][:200]}")

# Wait for deployment
print("\nWaiting for deployment...")
for i in range(40):
    try:
        ep = w.serving_endpoints.get(name=ENDPOINT_NAME)
        ready = str(ep.state.ready)
        config = str(ep.state.config_update)
        current_v = ep.config.served_entities[0].entity_version if ep.config and ep.config.served_entities else '?'
        print(f"  [{i*15}s] serving=v{current_v} ready={ready} config={config}")
        if ready == 'READY' and config == 'NOT_UPDATING':
            print(f"Endpoint READY (v{current_v})!")
            break
        if 'FAILED' in config:
            print("Deployment failed — check Serving UI for details.")
            break
    except Exception as e:
        print(f"  {e}")
    time.sleep(15)

In [ ]:
# Enable inference table logging via AI Gateway
try:
    r = api_call('PUT', f'/api/2.0/serving-endpoints/{ENDPOINT_NAME}/ai-gateway', {
        'inference_table_config': {
            'catalog_name': CATALOG,
            'schema_name': SCHEMA,
            'table_name_prefix': 'ipo_endpoint',
            'enabled': True,
        },
    })
    if 'error' not in r:
        print("Inference table enabled via AI Gateway")
        print(f"Table: {CATALOG}.{SCHEMA}.ipo_endpoint_payload")
    else:
        print(f"AI Gateway note: {r.get('error', '')[:200]}")
except Exception as e:
    print(f"Inference table setup: {e}")

## B. Test via REST API

`w.serving_endpoints.query()` sends a single request to the live endpoint — the same HTTP call that any external client would make. The payload is a list of records (`dataframe_records`); each record is a dictionary matching the model's input schema.

The `input` key maps to the `ChatMessage` field defined in the agent's MLflow input schema (set during registration in Lab 03). The response is a `QueryEndpointResponse` object; `.as_dict()` converts it to a plain Python dict for inspection.

In [ ]:
try:
    response = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=[{"input": "What are Snowflake's key risk factors?"}],
    )
    
    print("=== Endpoint Response ===")
    print(json.dumps(response.as_dict(), indent=2))
except Exception as e:
    print(f'Endpoint operation skipped: {str(e)[:200]}')
    print('This is expected if the endpoint is not deployed (trial workspace limitation).')


## C. A/B Testing with TrafficConfig

**A/B testing** in Model Serving routes a percentage of live traffic to each model version. This lets teams validate new versions incrementally — if v2 performs worse, traffic can be shifted back to v1 with no downtime.

Key rules:
- **Traffic percentages must sum to exactly 100.** The SDK will reject any configuration that does not sum to 100.
- Each route references a `served_model_name`, which corresponds to the `name` field of a `ServedEntityInput` (auto-generated from the entity name and version if not specified).
- `TrafficConfig` + `Route` live in `databricks.sdk.service.serving`.

The block below demonstrates the configuration pattern. It is wrapped in a `try/except` because the second served entity (`v2`) may not exist in the workspace — that is expected. The goal is to understand the pattern, not to require two real model versions.

In [ ]:
try:
    from databricks.sdk.service.serving import TrafficConfig, Route
    
    # A/B split: 50% v1, 50% v2
    # Traffic percentages MUST sum to 100 — the API will reject any other total.
    ab_traffic = TrafficConfig(
        routes=[
            Route(
                served_model_name=f"{MODEL_NAME.replace('.', '-')}-1",
                traffic_percentage=50,
            ),
            Route(
                served_model_name=f"{MODEL_NAME.replace('.', '-')}-2",
                traffic_percentage=50,
            ),
        ]
    )
    
    print("=== A/B Traffic Config (pattern reference) ===")
    print(f"Route 1: {ab_traffic.routes[0].served_model_name} → {ab_traffic.routes[0].traffic_percentage}%")
    print(f"Route 2: {ab_traffic.routes[1].served_model_name} → {ab_traffic.routes[1].traffic_percentage}%")
    print(f"Total  : {sum(r.traffic_percentage for r in ab_traffic.routes)}%  (must equal 100)")
    print()
    
    # Attempt to apply — will fail if v2 does not exist. That is expected.
    try:
        w.serving_endpoints.update_config(
            name=ENDPOINT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version="1",
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                ),
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version="2",
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                ),
            ],
            traffic_config=ab_traffic,
        )
        print("A/B split applied successfully.")
    except Exception as e:
        print(f"A/B update skipped (expected if v2 does not exist): {type(e).__name__}")
        print()
        print("This is expected. The pattern is what matters:")
        print("  - Define two ServedEntityInput entries (v1 and v2)")
        print("  - Create a TrafficConfig with two Routes summing to 100%")
        print("  - Call update_config() to apply with zero downtime")
except Exception as e:
    print(f'A/B traffic config note: {str(e)[:200]}')
    print('A/B routing pattern demonstrated above — actual deployment requires both model versions served.')


## D. Batch Inference with ai_query()

For batch workloads — scoring a table of questions in parallel — **SQL `ai_query()`** is the production pattern. It runs natively inside the warehouse, distributing calls across workers automatically. No Python loop, no driver overhead.

Requirements:
- The endpoint must be in `READY` state before `ai_query()` can route to it.
- The second argument to `ai_query()` is the column (or expression) to send as the prompt.
- Results are returned as strings and can be written directly to a Delta table.

We create a small `test_questions` table, run batch inference over it, and display the results.

In [ ]:
# Create a small test_questions table
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.test_questions AS
SELECT * FROM VALUES
  (1, 'What are Snowflake''s key risk factors?'),
  (2, 'How does Coinbase generate revenue?'),
  (3, 'What was DoorDash''s total addressable market claim in its S-1?'),
  (4, 'Summarise Palantir''s government contract strategy.'),
  (5, 'What manufacturing risks did Rivian disclose?')
AS t(id, question)
""")

count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.test_questions").first()[0]
print(f"Created {CATALOG}.{SCHEMA}.test_questions with {count} rows")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.test_questions"))

In [ ]:
try:
    # Batch inference: ai_query() over every row — runs in parallel across warehouse workers
    # Requires the endpoint to be in READY state.
    display(spark.sql(f"""
    SELECT
        id,
        question,
        ai_query('{ENDPOINT_NAME}', question) AS agent_answer
    FROM {CATALOG}.{SCHEMA}.test_questions
    ORDER BY id
    """))
except Exception as e:
    print(f'Skipped (endpoint not deployed): {str(e)[:200]}')


## E. The Signature Query

The signature query the VP asked for at the start of the project:

> *"Show me the clarity scores of the top 10 performing tech IPO stocks in their first year."*

We run it two ways:

1. **Via the deployed endpoint** — the agent receives a natural-language question, reasons about which tools to call, invokes `query_scored_database('DESC', 10)`, and returns a structured answer.
2. **Direct SQL** — what the agent's `query_scored_database` tool executes internally. This is the ground-truth result the agent's answer should match.

Both paths should converge on the same data: top-10 tickers by `twelve_month_return_pct`, joined with their `score_json` from `clarity_scores`.

In [ ]:
try:
    SIGNATURE_QUERY = (
        "Show me the clarity scores of the top 10 performing tech IPO stocks in their first year."
    )
    
    # --- Path 1: Via agent endpoint ---
    result = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=[{"input": SIGNATURE_QUERY}],
    )
    print("Agent response:")
    print(json.dumps(result.as_dict(), indent=2))
except Exception as e:
    print(f'Endpoint operation skipped: {str(e)[:200]}')
    print('This is expected if the endpoint is not deployed (trial workspace limitation).')


In [ ]:
# --- Path 2: Direct SQL (what the agent's query_scored_database tool does internally) ---
print("Direct SQL result (ground truth):")
display(spark.sql(f"""
SELECT s.company, s.ticker, s.twelve_month_return_pct, c.score_json
FROM {CATALOG}.{SCHEMA}.stock_performance s
JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
ORDER BY s.twelve_month_return_pct DESC
LIMIT 10
"""))

## Before / After

**Before this lab:**
- Agent runs only inside a Databricks notebook, requiring a live cluster
- Each query needs a developer to open a notebook and run cells manually
- Batch scoring requires a Python loop — slow and sequential
- The signature query exists as a SQL statement but has no public interface

**After this lab:**
- Agent is live as a REST endpoint: any HTTP client can query it
- `w.serving_endpoints.query()` demonstrates the programmatic client pattern
- `ai_query()` enables parallel batch inference directly from SQL
- The signature query runs end-to-end through the deployed agent
- A/B traffic split pattern is understood for future version rollout

In [ ]:
# Return key results via notebook exit (enables API result inspection)
import json as _json
_results = {}
try:
    _results["lab"] = "07-deployment"
    _results["status"] = "completed"
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")

In [ ]:
# --- CLEANUP (commented out — uncomment to delete the serving endpoint) ---
# WARNING: This permanently deletes the endpoint. All clients will lose access.

# w.serving_endpoints.delete(name=ENDPOINT_NAME)
# print(f"Deleted endpoint: {ENDPOINT_NAME}")